# Day 7: Finite State Machines

## Pre-Class Videos (~50 minutes total)

| # | Segment | Duration | File |
|---|---------|----------|------|
| 1 | FSM Theory & Architecture | ~12 min | `d07_s1_fsm_theory_architecture.html` |
| 2 | The 3-Always-Block Template | ~15 min | `d07_s2_three_block_template.html` |
| 3 | State Encoding | ~8 min | `d07_s3_state_encoding.html` |
| 4 | FSM Design Methodology | ~15 min | `d07_s4_fsm_design_methodology.html` |

## Code Examples

| File | Description |
|------|-------------|
| `code/day07_ex01_fsm_template.v` | Complete 3-block traffic light FSM with self-checking testbench |
| `code/day07_ex02_pattern_detector.v` | "101" sequence detector (Moore) with overlap support, self-test |

## Diagrams

| File | Description |
|------|-------------|
| `diagrams/d07_fsm_block_diagram.svg` | 3-block FSM architecture: state register, next-state, output logic |
| `diagrams/d07_traffic_light_states.svg` | Traffic light state diagram: GREEN → YELLOW → RED → GREEN |

## Key Concepts
- FSM = states + transitions + outputs
- Moore (outputs = f(state)) vs. Mealy (outputs = f(state, inputs))
- 3-block template: state register, next-state logic, output logic
- State encoding: binary (default), one-hot (FPGAs), Gray (CDC)
- Design methodology: states → transitions → outputs → diagram → code → test

## Directory Structure

```
day07_finite_state_machines/
├── d07_s1_fsm_theory_architecture.html
├── d07_s2_three_block_template.html
├── d07_s3_state_encoding.html
├── d07_s4_fsm_design_methodology.html
├── code/
│   ├── day07_ex01_fsm_template.v
│   └── day07_ex02_pattern_detector.v
├── diagrams/
│   ├── d07_fsm_block_diagram.svg
│   └── d07_traffic_light_states.svg
├── day07_quiz.md
└── day07_readme.md
```

---
## Code Examples

### `day07_ex01_fsm_template.v`

```verilog
// =============================================================================
// day07_ex01_fsm_template.v — 3-Always-Block FSM Template (Traffic Light)
// Day 7: Finite State Machines
// Accelerated HDL for Digital System Design · UCF ECE
// =============================================================================
// The canonical FSM coding style:
//   Block 1 (seq):  State register — just a flip-flop
//   Block 2 (comb): Next-state logic — case statement with default
//   Block 3 (comb): Output logic — Moore: outputs depend only on state
// =============================================================================
// Build:  iverilog -DSIMULATION -o sim day07_ex01_fsm_template.v && vvp sim
// View:   gtkwave tb_traffic_light.vcd
// Synth:  yosys -p "read_verilog day07_ex01_fsm_template.v; synth_ice40 -top traffic_light"
// =============================================================================

module traffic_light #(
    `ifdef SIMULATION
    parameter GREEN_TIME  = 10,    // Short for simulation
    parameter YELLOW_TIME = 4,
    parameter RED_TIME    = 10
    `else
    parameter GREEN_TIME  = 125_000_000,  // 5 sec at 25 MHz
    parameter YELLOW_TIME =  25_000_000,  // 1 sec
    parameter RED_TIME    = 125_000_000   // 5 sec
    `endif
)(
    input  wire i_clk,
    input  wire i_reset,
    output reg  o_red,
    output reg  o_yellow,
    output reg  o_green
);

    // ---- State Encoding ----
    localparam S_GREEN  = 2'd0;
    localparam S_YELLOW = 2'd1;
    localparam S_RED    = 2'd2;

    reg [1:0] r_state, r_next_state;

    // ---- Timer ----
    localparam MAX_TIME = (GREEN_TIME > RED_TIME) ? GREEN_TIME : RED_TIME;
    reg [$clog2(MAX_TIME)-1:0] r_timer;
    wire w_timer_done;

    // Timer target depends on current state
    reg [$clog2(MAX_TIME)-1:0] r_timer_target;

    always @(*) begin
        case (r_state)
            S_GREEN:  r_timer_target = GREEN_TIME - 1;
            S_YELLOW: r_timer_target = YELLOW_TIME - 1;
            S_RED:    r_timer_target = RED_TIME - 1;
            default:  r_timer_target = GREEN_TIME - 1;
        endcase
    end

    assign w_timer_done = (r_timer == r_timer_target);

    // Timer: reset on state change, else increment
    always @(posedge i_clk) begin
        if (i_reset || (r_state != r_next_state))
            r_timer <= 0;
        else if (!w_timer_done)
            r_timer <= r_timer + 1;
    end

    // ============================================================
    // Block 1 — State Register (Sequential)
    // Always this simple. No logic here.
    // ============================================================
    always @(posedge i_clk) begin
        if (i_reset)
            r_state <= S_GREEN;
        else
            r_state <= r_next_state;
    end

    // ============================================================
    // Block 2 — Next-State Logic (Combinational)
    // Critical: default assignment prevents latches.
    // ============================================================
    always @(*) begin
        r_next_state = r_state;  // DEFAULT: stay in current state

        case (r_state)
            S_GREEN: begin
                if (w_timer_done)
                    r_next_state = S_YELLOW;
            end

            S_YELLOW: begin
                if (w_timer_done)
                    r_next_state = S_RED;
            end

            S_RED: begin
                if (w_timer_done)
                    r_next_state = S_GREEN;
            end

            default: r_next_state = S_GREEN;  // illegal state recovery
        endcase
    end

    // ============================================================
    // Block 3 — Output Logic (Combinational — Moore)
    // Outputs depend ONLY on r_state.
    // ============================================================
    always @(*) begin
        // Defaults: all off
        o_red    = 1'b0;
        o_yellow = 1'b0;
        o_green  = 1'b0;

        case (r_state)
            S_GREEN:  o_green  = 1'b1;
            S_YELLOW: o_yellow = 1'b1;
            S_RED:    o_red    = 1'b1;
            default:  o_red    = 1'b1;  // safe default
        endcase
    end

endmodule

// =============================================================================
// Self-Checking Testbench
// =============================================================================
`ifdef SIMULATION
module tb_traffic_light;
    reg  clk = 0, reset = 1;
    wire red, yellow, green;

    traffic_light uut (
        .i_clk(clk), .i_reset(reset),
        .o_red(red), .o_yellow(yellow), .o_green(green)
    );

    always #20 clk = ~clk;  // 25 MHz

    integer test_count = 0, fail_count = 0;

    task check_state;
        input exp_r, exp_y, exp_g;
        input [8*20-1:0] name;
    begin
        test_count = test_count + 1;
        if (red !== exp_r || yellow !== exp_y || green !== exp_g) begin
            $display("FAIL: %0s — R=%b Y=%b G=%b (expected R=%b Y=%b G=%b)",
                     name, red, yellow, green, exp_r, exp_y, exp_g);
            fail_count = fail_count + 1;
        end else
            $display("PASS: %0s — R=%b Y=%b G=%b", name, red, yellow, green);
    end
    endtask

    task wait_cycles;
        input integer n;
        integer i;
        for (i = 0; i < n; i = i + 1) @(posedge clk);
    endtask

    initial begin
        $dumpfile("tb_traffic_light.vcd");
        $dumpvars(0, tb_traffic_light);

        $display("\n=== Traffic Light FSM Testbench ===\n");

        // Reset
        wait_cycles(3);
        reset = 0;
        @(posedge clk); #1;
        check_state(0, 0, 1, "After reset: GREEN");

        // Wait for GREEN → YELLOW transition (GREEN_TIME = 10 cycles)
        wait_cycles(10);
        @(posedge clk); #1;
        check_state(0, 1, 0, "GREEN→YELLOW");

        // Wait for YELLOW → RED transition (YELLOW_TIME = 4 cycles)
        wait_cycles(4);
        @(posedge clk); #1;
        check_state(1, 0, 0, "YELLOW→RED");

        // Wait for RED → GREEN transition (RED_TIME = 10 cycles)
        wait_cycles(10);
        @(posedge clk); #1;
        check_state(0, 0, 1, "RED→GREEN (cycle 2)");

        // Test mid-cycle reset
        wait_cycles(5);
        reset = 1;
        @(posedge clk); #1;
        reset = 0;
        @(posedge clk); #1;
        check_state(0, 0, 1, "Mid-cycle reset → GREEN");

        // Summary
        $display("\n=== TEST SUMMARY ===");
        $display("Tests run:    %0d", test_count);
        $display("Tests passed: %0d", test_count - fail_count);
        $display("Tests failed: %0d", fail_count);
        if (fail_count == 0)
            $display("\n*** ALL TESTS PASSED ***\n");
        else
            $display("\n*** %0d FAILURES ***\n", fail_count);
        $finish;
    end
endmodule
`endif
```

### `day07_ex02_pattern_detector.v`

```verilog
// =============================================================================
// day07_ex02_pattern_detector.v — Sequence Detector FSM (Moore)
// Day 7: Finite State Machines
// Accelerated HDL for Digital System Design · UCF ECE
// =============================================================================
// Detects the serial bit pattern "101" on i_serial_in.
// When the full pattern is recognized, o_detected pulses high for one state.
// Overlapping detection supported (e.g., 10101 detects twice).
// =============================================================================
// Build:  iverilog -DSIMULATION -o sim day07_ex02_pattern_detector.v && vvp sim
// Synth:  yosys -p "read_verilog day07_ex02_pattern_detector.v; synth_ice40 -top pattern_detector"
// =============================================================================

module pattern_detector (
    input  wire i_clk,
    input  wire i_reset,
    input  wire i_serial_in,
    output reg  o_detected
);

    // ---- State Encoding ----
    localparam S_IDLE  = 2'd0;   // Waiting for '1'
    localparam S_GOT1  = 2'd1;   // Seen '1'
    localparam S_GOT10 = 2'd2;   // Seen '10'
    localparam S_MATCH = 2'd3;   // Seen '101' — detect!

    reg [1:0] r_state, r_next_state;

    // ============================================================
    // Block 1 — State Register
    // ============================================================
    always @(posedge i_clk) begin
        if (i_reset)
            r_state <= S_IDLE;
        else
            r_state <= r_next_state;
    end

    // ============================================================
    // Block 2 — Next-State Logic
    // ============================================================
    always @(*) begin
        r_next_state = r_state;  // default: stay

        case (r_state)
            S_IDLE: begin
                if (i_serial_in)
                    r_next_state = S_GOT1;
            end

            S_GOT1: begin
                if (!i_serial_in)
                    r_next_state = S_GOT10;
                // else stay in S_GOT1 (still seeing 1s)
            end

            S_GOT10: begin
                if (i_serial_in)
                    r_next_state = S_MATCH;  // "101" complete!
                else
                    r_next_state = S_IDLE;   // "100" — restart
            end

            S_MATCH: begin
                // Overlap: after "101", the final '1' could start a new pattern
                if (i_serial_in)
                    r_next_state = S_GOT1;   // overlap: "101" → "1" of next
                else
                    r_next_state = S_GOT10;  // overlap: "101" → "10" of next
            end

            default: r_next_state = S_IDLE;
        endcase
    end

    // ============================================================
    // Block 3 — Output Logic (Moore)
    // ============================================================
    always @(*) begin
        o_detected = 1'b0;   // default

        case (r_state)
            S_MATCH: o_detected = 1'b1;
            default: o_detected = 1'b0;
        endcase
    end

endmodule

// =============================================================================
// Self-Checking Testbench
// =============================================================================
`ifdef SIMULATION
module tb_pattern_detector;
    reg  clk = 0, reset = 1, serial_in = 0;
    wire detected;

    pattern_detector uut (
        .i_clk(clk), .i_reset(reset),
        .i_serial_in(serial_in), .o_detected(detected)
    );

    always #20 clk = ~clk;

    integer test_count = 0, fail_count = 0;

    task send_bit;
        input bit_val;
    begin
        serial_in = bit_val;
        @(posedge clk); #1;
    end
    endtask

    task check_detect;
        input exp;
        input [8*30-1:0] name;
    begin
        test_count = test_count + 1;
        if (detected !== exp) begin
            $display("FAIL: %0s — detected=%b (expected %b)", name, detected, exp);
            fail_count = fail_count + 1;
        end else
            $display("PASS: %0s — detected=%b", name, detected);
    end
    endtask

    initial begin
        $dumpfile("tb_pattern_detector.vcd");
        $dumpvars(0, tb_pattern_detector);

        $display("\n=== Pattern Detector FSM Testbench ===");
        $display("    Detecting: '101'\n");

        // Reset
        @(posedge clk); @(posedge clk);
        reset = 0;
        @(posedge clk); #1;

        // Test 1: Simple "101" detection
        send_bit(1);
        check_detect(0, "After '1'");
        send_bit(0);
        check_detect(0, "After '10'");
        send_bit(1);
        check_detect(0, "After '101' (detect next cycle)");
        @(posedge clk); #1;
        check_detect(1, "'101' DETECTED");

        // Test 2: "110" — no detection
        reset = 1; @(posedge clk); reset = 0; @(posedge clk); #1;
        send_bit(1); send_bit(1); send_bit(0);
        @(posedge clk); #1;
        check_detect(0, "'110' — no detect");

        // Test 3: Overlapping "10101" — should detect twice
        reset = 1; @(posedge clk); reset = 0; @(posedge clk); #1;
        send_bit(1); send_bit(0); send_bit(1);
        @(posedge clk); #1;
        check_detect(1, "First '101' in '10101'");
        send_bit(0); send_bit(1);
        @(posedge clk); #1;
        check_detect(1, "Second '101' (overlap)");

        // Summary
        $display("\n=== TEST SUMMARY ===");
        $display("Tests: %0d  Passed: %0d  Failed: %0d",
                 test_count, test_count - fail_count, fail_count);
        if (fail_count == 0)
            $display("\n*** ALL TESTS PASSED ***\n");
        else
            $display("\n*** %0d FAILURES ***\n", fail_count);
        $finish;
    end
endmodule
`endif
```

---
## Pre-Class Self-Check Quiz

**Q1:** What are the three blocks in the 3-always-block FSM coding style, and what type of logic is each?

<details><summary>Answer</summary>

1. **State Register** — sequential (`always @(posedge clk)`) — just a flip-flop
2. **Next-State Logic** — combinational (`always @(*)`) — computes next state from current state + inputs
3. **Output Logic** — combinational (`always @(*)`) — computes outputs from current state (Moore) or state + inputs (Mealy)

</details>

**Q2:** What is the critical first line in the next-state logic block and why?

<details><summary>Answer</summary>

`r_next_state = r_state;` — This default assignment means "if no transition condition fires, stay in the current state." It prevents latch inference (every path assigns `r_next_state`) and handles the common case of remaining in the current state.

</details>

**Q3:** What is the difference between Moore and Mealy machines? Which do we default to?

<details><summary>Answer</summary>

**Moore**: outputs depend only on state — outputs are stable and change only on clock edges.
**Mealy**: outputs depend on state AND current inputs — can react within the same clock cycle but may produce glitches.
We default to **Moore** because it's safer and easier to debug.

</details>

**Q4:** Name three common FSM coding mistakes.

<details><summary>Answer</summary>

1. No default assignment in next-state logic → latch on `r_next_state`
2. Forgetting output defaults in Block 3 → latches on outputs
3. No `default` case → FSM gets stuck in illegal state
4. Using blocking `=` in the state register (Block 1) → should be `<=`
5. Putting logic in Block 1 instead of keeping it as a simple flip-flop

(Any three of these.)

</details>